# Submission: final mlp_4out_emb_bn

Report: [mlp_report.md](../../mlp_report.md#final-comparison)


In [1]:
import os
import sys

import numpy as np
import torch

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.modeling import MeterMLP, make_loaders
from ashrae.submission import build_submission, check_submission, to_readings
from ashrae.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
CHUNK_SIZE = 1_000_000  # test rows predicted at once
EMB_DIM = 12  # exp03
WIDTH, DROPOUT, N_LAYERS = 64, 0.1, 2  # exp15, exp16
LR = 1e-2  # exp07; divided by 10 at half and three quarters of training (exp17)
if SMOKE:
    EXPERIMENT = "ashrae-submission-smoke"
    N_TRAIN = 20_000
    N_TEST = 50_000
    EPOCHS = 4  # long enough for both lr drops
    SUBMISSION_PATH = None  # partial test set, so nothing to submit
else:
    EXPERIMENT = "ashrae-submission"
    N_TRAIN = 2_000_000  # the rows per epoch the schedule was tuned on, now spread over all of 2016
    N_TEST = None  # all 41.7M test rows
    EPOCHS = 40
    SUBMISSION_PATH = PROCESSED / "submission_mlp_final.csv"

experiment = Experiment(EXPERIMENT, "submission_final")

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: sqlite:////Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/mlflow.db, experiment 'ashrae-submission', session 2026-10-13 11:09:06


## Training rows


In [2]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
train_df = df[~df.anomaly].sample(
    N_TRAIN, random_state=0
)  # all of 2016, anomalies dropped
del df

year_median = train_df.year_built.median()  # reused for the test set
X_train = build_features(train_df, year_median)
train_loader, _, scaler = make_loaders(train_df, None, X_train, None)
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features")

2,000,000 training rows, 33 features


## Model


In [3]:
def step_decay(params):
    """Adam at LR, divided by 10 at epochs 20 and 30 of 40."""
    opt = torch.optim.Adam(params, lr=LR)
    sched = torch.optim.lr_scheduler.MultiStepLR(
        opt, milestones=[EPOCHS // 2, EPOCHS * 3 // 4], gamma=0.1
    )
    return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}


model, run_id = experiment.fit(
    lambda: MeterMLP(
        X_train.shape[1],
        n_buildings,
        emb_dim=EMB_DIM,
        batch_norm=True,
        dropout=DROPOUT,
        width=WIDTH,
        n_layers=N_LAYERS,
    ),
    train_loader,
    run_name="mlp_4out_emb_bn | final | all 2016",
    epochs=EPOCHS,
    lr=LR,
    optimizer=step_decay,
    params={
        "n_train": N_TRAIN,
        "emb_dim": EMB_DIM,
        "width": WIDTH,
        "dropout": DROPOUT,
        "n_layers": N_LAYERS,
        "schedule": "step",
    },
    tags={"model": "mlp_4out_emb_bn", "schedule": "step"},
)
model = model.eval()
print(
    f"final mlp_4out_emb_bn: training RMSLE {experiment.history(run_id, 'train_rmsle')[-1]:.3f}"
)

/Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
mlp_4out_emb_bn | final | all 2016: 100%|██████████| 19560/19560 [01:28<00:00, 220.52batch/s]


final mlp_4out_emb_bn: training RMSLE 0.819


## Test predictions


In [4]:
test_df = read_processed("test_merged.parquet")
if N_TEST:
    test_df = test_df.head(N_TEST)


@torch.no_grad()
def predict_log(rows):
    """Clipped log1p predictions for `rows`, built with the same feature columns and scaler as training."""
    x = scaler.transform(
        build_features(rows, year_median).reindex(columns=X_train.columns, fill_value=0)
    )
    batch = {
        "x": torch.tensor(x, dtype=torch.float32, device=model.device),
        "meter": torch.tensor(
            rows.meter.to_numpy(), dtype=torch.long, device=model.device
        ),
        "building": torch.tensor(
            rows.building_id.to_numpy(), dtype=torch.long, device=model.device
        ),
    }
    return model(batch).clamp(min=0).cpu().numpy()


starts = range(0, len(test_df), CHUNK_SIZE)
mlp_log = np.concatenate(
    [predict_log(test_df.iloc[start : start + CHUNK_SIZE]) for start in starts]
)
print(f"mean log1p prediction: {mlp_log.mean():.3f}")

mean log1p prediction: 4.170


In [5]:
reading = to_readings(mlp_log, test_df)  # site 0 electricity back in kBTU
submission = build_submission(test_df, reading)
if SUBMISSION_PATH:
    check_submission(submission)
    submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.4f")
    print(f"saved {SUBMISSION_PATH}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

saved /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Homeworks/Homework 1 MLP/data/processed/submission_mlp_final.csv: 41,697,600 rows


,meter_reading
count,41697600.00
mean,361.80
std,2331.61
min,0.00
25%,20.44
50%,76.31
75%,232.63
max,334490.09


In [7]:
if SUBMISSION_PATH:  # multithreaded 7-Zip CLI (brew install sevenzip), much faster than py7zr
    !cd "{SUBMISSION_PATH.parent}" && rm -f "{SUBMISSION_PATH.stem}.7z" && 7zz a -mmt=on "{SUBMISSION_PATH.stem}.7z" "{SUBMISSION_PATH.name}"


7-Zip (z) 26.04 (arm64) : Copyright (c) 1999-2026 Igor Pavlov : 2026-10-05
 64-bit arm_v:8.5-A locale=C.UTF-8 Threads:14 OPEN_MAX:1048575, ASM

Scanning the drive:
  0M Sca        1 file, 711921801 bytes (679 MiB)

Creating archive: submission_mlp_final.7z

Add new data to archive: 1 file, 711921801 bytes (679 MiB)

      0% + submission_mlp_final.cs                                1% 1 + submission_mlp_final.cs                                  2% 1 + submission_mlp_final.cs                                  3% 1 + submission_mlp_final.cs                                  4% 1 + submission_mlp_final.cs                                  5% 1 + submission_mlp_final.cs                                  6% 1 + submission_mlp_final.cs                                  7% 1 + submission_mlp_final.cs                                  8% 1 + submission_mlp_final.cs                                  9% 1 + submission_mlp_final.cs                                 10% 1 + submission_mlp_final.cs         